In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from time import time
import hexaly.optimizer as hexaly
from itertools import product
import matplotlib.pyplot as plt
import sys
import pickle

In [2]:
start_time = time()

In [ ]:
hexaly.HxVersion.license_content = "LICENSE_KEY = ED3A-2222-89F4B124-770D-60A55B936308D780-9506208B36204986-9B3E-E289-C66E"

In [4]:
# with open('x.pkl', 'rb') as f:
#     x_initial_cg_cplex = pickle.load(f)

# # print(x_initial_cg_cplex)

In [5]:
def display_callback(optimizer, event_type):
    if event_type == hexaly.HxCallbackType.DISPLAY:
        solution = optimizer.solution
        if optimizer.model.nb_objectives > 0:
            objective_expr = optimizer.model.objectives[0]
            objective_value = solution.get_value(objective_expr)
            objective_bound = solution.get_objective_bound(0)
            optimality_gap = solution.get_objective_gap(0)
            print(f"Objective Value: {objective_value}")
            print(f"Objective Bound: {objective_bound}")
            print(f"Optimality Gap: {optimality_gap}")
        else:
            print("No objectives defined in the model.")

# Defining the speed processing of each type of station

In [6]:
#GLOBAL VARIABLES
STATIC_SPEED = 37700
PALETTE_SPEED = 57200
DYNAMIC_SPEED = 83200

# Reduce memory function on data frames

In [7]:
#function  to reduce the memory usage
def reduce_mem_usage(df, verbose=False):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

# Read client file and clean the data

In [8]:
data_df_initial= pd.read_csv(r"C:\ermal\notebooks\Correlated_Storage_Assignment_Problem\Data\BERNER_ORDER_LINES_09-12.csv", sep=';')[['PRODUCT','ORDER','STATION']].drop_duplicates().dropna()#, nrows=20000)

#Calculate the unique number of product per station
unique_counts = data_df_initial[['PRODUCT','STATION']].groupby('PRODUCT')['STATION'].nunique().reset_index(name='numb_stat_per_prod')
#Merging the count back to the original dataframe
data_df_initial = data_df_initial.merge(unique_counts, on='PRODUCT', how='left')

#Updating the product assignment with the last station they correspond
# Step 1: Identify products with numb_stat_per_prod > 1.
products_with_multiple_stations = data_df_initial[data_df_initial['numb_stat_per_prod'] > 1]['PRODUCT'].unique()

# Step 2: For these products, find the maximum order ID and the corresponding station ID.
max_order_stations = data_df_initial[data_df_initial['PRODUCT'].isin(products_with_multiple_stations)] \
    .sort_values(by='ORDER', ascending=False) \
    .drop_duplicates(subset=['PRODUCT'], keep='first')[['PRODUCT', 'STATION']]

# Step 3: Update the station ID for all occurrences of these products.
# First, merge this information back to the original df_order.
data_df_initial = data_df_initial.merge(max_order_stations, on='PRODUCT', how='left', suffixes=('', '_max_order'))

# If STATION_max_order is not NaN (i.e., for products with multiple stations), update the STATION column.
data_df_initial['STATION'] = data_df_initial.apply(lambda x: x['STATION_max_order'] if pd.notna(x['STATION_max_order']) else x['STATION'], axis=1)

# Drop the temporary STATION_max_order column as it's no longer needed.
data_df_initial.drop(columns=['STATION_max_order', 'numb_stat_per_prod'], inplace=True)

data_df_new = data_df_initial.copy()

# This is a filetering of data for testing with CG

In [9]:
#data_df_filter = data_df_new.copy()
data_df_new['Product_Frequency'] = data_df_new.groupby('PRODUCT')['ORDER'].transform('count')
product_test_list = np.unique(data_df_new[data_df_new['Product_Frequency']>2000]['PRODUCT'])
#Remove orders that contain only one product
# Calculate the number of unique 'PRODUCT' values for each 'ORDER'
order_product_counts = data_df_new.groupby('ORDER')['PRODUCT'].nunique()

# Filter 'ORDER's where the count of unique 'PRODUCT' values is more than 3
orders_to_keep = order_product_counts[order_product_counts > 30].index

# Filter the original DataFrame to keep only those rows with 'ORDER's in 'orders_to_keep'
data_df_new = data_df_new[data_df_new['ORDER'].isin(orders_to_keep)]
data_df_new = data_df_new[data_df_new['PRODUCT'].isin(product_test_list)]

# Remove stations as required and filterout orders that have only 1 product

In [10]:
stations_not_included = ['01.Z8','01.15','01.GED']
#Filter and count occurrences at the same time
data_df_new = data_df_new[~(data_df_new['STATION'].isin(stations_not_included))]
data_df_new.loc[data_df_new['STATION'] == '01.GE4', 'STATION'] = '01.E4'
# Calculate the total number of unique orders for setting the threshold
total_unique_orders = data_df_new['ORDER'].nunique()

# Count the number of 'Location_ID' for each STATION
#Calculate the unique number of product per station
total_number_of_location_per_station = data_df_new.groupby('STATION')['PRODUCT'].nunique().to_dict()


# Count the number of initial lines for each STATION
#Calculate number of product per station
station_lines_initial = data_df_new.groupby('STATION')['PRODUCT'].count().to_dict()

#Calculate the frequency for each product
#data_df_new['Product_Frequency'] = data_df_new.groupby('PRODUCT')['ORDER'].transform('count')


stations = data_df_new['STATION'].unique().astype(str)

## Map Station IDs

In [11]:
def map_station_name(df_order):
    #change statin names to presere confidenciality
    unique_stations = df_order['STATION'].unique()
    mapping_dict = {station: f'S_{i}' for i, station in enumerate(unique_stations, start=1)}
    df_order['NewStationID'] = df_order['STATION'].map(mapping_dict)
    print(mapping_dict)
    return df_order, mapping_dict

In [12]:
data_df_new,mapping_dict = map_station_name(data_df_new)

{'01.24': 'S_1', '01.13': 'S_2', '01.10': 'S_3', '01.27': 'S_4', '01.08': 'S_5', '01.09': 'S_6', '01.07': 'S_7', '01.06': 'S_8', '01.23': 'S_9', '01.12': 'S_10', '01.11': 'S_11', '01.22': 'S_12', '01.25': 'S_13'}


In [13]:
stations_mapped = data_df_new['NewStationID'].unique().astype(str)

In [14]:
total_number_of_location_per_station_mapped = data_df_new.groupby('NewStationID')['PRODUCT'].nunique().to_dict()

# Categorise stations based on their type

In [15]:
static_stations = ['01.E4','01.31','01.30']
palette_stations = ['01.01','01.02','01.03','01.04','01.05']
union_stat = static_stations + palette_stations
dynamic_stations = np.array(data_df_new[~(data_df_new['STATION'].isin(union_stat))]['STATION'].drop_duplicates())

In [16]:
print(total_number_of_location_per_station)

{'01.06': 6, '01.07': 5, '01.08': 9, '01.09': 3, '01.10': 4, '01.11': 3, '01.12': 2, '01.13': 6, '01.22': 1, '01.23': 2, '01.24': 2, '01.25': 1, '01.27': 2}


# Specify the speed of processing on each station based on their type and the numbre of locations they have

In [17]:
speed = {s:STATIC_SPEED/total_number_of_location_per_station[s] for s in static_stations if s in stations}
speed.update({s:PALETTE_SPEED/total_number_of_location_per_station[s] for s in palette_stations if s in stations})
speed.update({s:DYNAMIC_SPEED/total_number_of_location_per_station[s] for s in dynamic_stations if s in stations})

In [18]:
print(len(data_df_new))
print(len(np.unique(data_df_new['PRODUCT'].values)))
print(len(np.unique(data_df_new['ORDER'].values)))
print(len(np.unique(data_df_new['STATION'].values)))
print(sum(list(total_number_of_location_per_station.values())))

10130
46
3497
13
46


In [19]:
print(speed)

{'01.24': 41600.0, '01.13': 13866.666666666666, '01.10': 20800.0, '01.27': 41600.0, '01.08': 9244.444444444445, '01.09': 27733.333333333332, '01.07': 16640.0, '01.06': 13866.666666666666, '01.23': 41600.0, '01.12': 41600.0, '01.11': 27733.333333333332, '01.22': 83200.0, '01.25': 83200.0}


In [20]:
print(stations_mapped)

['S_1' 'S_2' 'S_3' 'S_4' 'S_5' 'S_6' 'S_7' 'S_8' 'S_9' 'S_10' 'S_11'
 'S_12' 'S_13']


In [21]:
speed_mapped = {mapping_dict[key]: value for key, value in speed.items()}

In [22]:
print(speed_mapped)

{'S_1': 41600.0, 'S_2': 13866.666666666666, 'S_3': 20800.0, 'S_4': 41600.0, 'S_5': 9244.444444444445, 'S_6': 27733.333333333332, 'S_7': 16640.0, 'S_8': 13866.666666666666, 'S_9': 41600.0, 'S_10': 41600.0, 'S_11': 27733.333333333332, 'S_12': 83200.0, 'S_13': 83200.0}


In [23]:
print(data_df_new)

           PRODUCT       ORDER STATION  Product_Frequency NewStationID
11180      1000381  8076574253   01.24               2826          S_1
11232      1000381  8076580403   01.24               2826          S_1
11257      1000381  8076582478   01.24               2826          S_1
11269      1000381  8076583176   01.24               2826          S_1
11274      1000381  8076583403   01.24               2826          S_1
...            ...         ...     ...                ...          ...
1462593  857386-10  8076733102   01.11               4154         S_11
1462630  857386-10  8076737080   01.11               4154         S_11
1462633  857386-10  8076737109   01.11               4154         S_11
1462652  857386-10  8076738163   01.11               4154         S_11
1462672  857386-10  8076739354   01.11               4154         S_11

[10130 rows x 5 columns]


# Fromat the input data in the form of product and the list of orders it is in

In [24]:
# data_df_new = pd.read_csv("..\Data\df_order_new.csv")[['Product','ORDER','MAJ_STAT']].drop_duplicates().dropna()
# data_df_new.rename(columns= {'Product': 'PRODUCT', 'MAJ_STAT':'STATION'}, inplace=True)
# data_df_new = reduce_mem_usage(data_df_new, verbose=False)

In [25]:
df_product = data_df_new.groupby('PRODUCT').agg({
    'ORDER': list,  # Aggregate orders into a list
    'STATION': 'first',  # Assuming each product is assigned to a unique station, we take the first occurrence
    'NewStationID': 'first',
    #'Product_Frequency': 'first'
}).reset_index().rename(columns={'ORDER': 'ORDER_LIST'})
df_product['Product_Frequency'] = df_product['ORDER_LIST'].apply(len)
# df_product = pd.merge(df_product, data_df_filter[['PRODUCT', 'Product_Frequency']].drop_duplicates(), 
#                      on='PRODUCT', how='left')
df_product = df_product.sort_values(by='Product_Frequency', ascending=False)
df_product.set_index('PRODUCT',inplace=True)
df_product.head(5)

,ORDER_LIST,STATION,NewStationID,Product_Frequency
PRODUCT,,,,
4375-100,"[8076384989, 8076385367, 8076385666, 807638572...",01.25,S_13,609
6467-100,"[8076385368, 8076385985, 8076385988, 807638690...",01.08,S_5,554
38278,"[8076384886, 8076385656, 8076386298, 807638670...",01.06,S_8,532
111733-100,"[8076385666, 8076386755, 8076386904, 807638839...",01.27,S_4,510
369964-100,"[8076385367, 8076385868, 8076385980, 807638598...",01.13,S_2,438


In [26]:
# #low_freq_prod = df_product[(df_product['Product_Frequency']<=4) & (df_product['STATION'].isin(static_stations))].index
# low_freq_prod = df_product[(df_product['Product_Frequency']<=179)].index
# df_product = df_product[~df_product.index.isin(low_freq_prod)]
# print(len(low_freq_prod))

# Dataframe the presents unique orders with the list of products they contain

In [27]:
data_df_new2 = data_df_new[['PRODUCT','ORDER']].drop_duplicates()
df_order = data_df_new2[data_df_new2['PRODUCT'].isin(df_product.index)].groupby('ORDER')['PRODUCT'].apply(list).reset_index(name='PRODUCT_LIST')
df_order.set_index('ORDER',inplace=True)

# Define the locations available on each station based on the products that will go through the main process of assignment

In [28]:
total_number_of_location_per_station_2 = df_product.groupby('STATION').size().to_dict()
print(total_number_of_location_per_station_2)
total_number_of_location_per_station_mapped_2 = df_product.groupby('NewStationID').size().to_dict()

{'01.06': 6, '01.07': 5, '01.08': 9, '01.09': 3, '01.10': 4, '01.11': 3, '01.12': 2, '01.13': 6, '01.22': 1, '01.23': 2, '01.24': 2, '01.25': 1, '01.27': 2}


In [29]:
# Convert df_product and speed data into dictionaries for quick lookup
product_frequency = (
    df_product[['Product_Frequency']]
    .set_index(df_product.index)['Product_Frequency']
    .to_dict()
)
# Assuming data_df_new contains PRODUCT and STATION mapping directly
product_to_station = (
    df_product[['STATION']]
    .set_index(df_product.index)['STATION']
    .to_dict()
)

In [30]:
print(len(product_frequency))
print(len(product_to_station))

46
46


# Provide the initital workload of each station that should be respected during the assignment from the solver as a constraint

In [31]:
products = df_product.index
orders = list(df_product['ORDER_LIST'].explode().unique())
data_group = data_df_new[data_df_new['PRODUCT'].isin(df_product.index)]

In [32]:
print(len(orders))

3497


In [33]:
total_length = {s:0 for s in stations}
count = {s:0 for s in stations}
for p in products:
    for s in stations:
        if product_to_station[p] == s:
                    total_length[s] += product_frequency[p] / speed[s]
                    count[s]+=1

print(total_length)

{np.str_('01.24'): 0.0049038461538461545, np.str_('01.13'): 0.09014423076923077, np.str_('01.10'): 0.02548076923076923, np.str_('01.27'): 0.018052884615384616, np.str_('01.08'): 0.20682692307692307, np.str_('01.09'): 0.02451923076923077, np.str_('01.07'): 0.06923076923076923, np.str_('01.06'): 0.13269230769230772, np.str_('01.23'): 0.0040384615384615385, np.str_('01.12'): 0.006418269230769231, np.str_('01.11'): 0.01979567307692308, np.str_('01.22'): 0.0026201923076923078, np.str_('01.25'): 0.007319711538461539}


In [34]:
for s in stations:    
    total_length[s] = 1.05*total_length[s]

In [35]:
print(sum(total_length[s] for s in stations))
print(sum(count[s] for s in stations))
print(sum(total_number_of_location_per_station_2[s] for s in stations))

0.6426454326923076
46
46


In [36]:
#Scale the workload coefficent to aviod numerical instabilities in the solver process
# Define scaling factors to normalize the right-hand side to 1.
# We also avoid division by zero by using a small tolerance.
scaling = { s: (1.0 / total_length[s] if total_length[s] > 1e-12 else 1.0) for s in stations }

In [37]:
scaling_mapped = {mapping_dict[key]: value for key, value in scaling.items()}


In [38]:
print(scaling_mapped)

{'S_1': 194.21101774042947, 'S_2': 10.565079365079365, 'S_3': 37.37646001796945, 'S_4': 52.755056749730514, 'S_5': 4.604724269996236, 'S_6': 38.8422035480859, 'S_7': 13.756613756613756, 'S_8': 7.177363699102827, 'S_9': 235.82766439909295, 'S_10': 148.38594613875512, 'S_11': 48.11056177176395, 'S_12': 363.47750109218, 'S_13': 130.11181484087888}


# Generate the smooth start solution to provide to the solver

In [39]:
def generate_mip_start_mapped(data_df):
    # Approach is to initialize initial_solution_product_station and initial_solution_order_station
    initial_solution_product_station = {}
    initial_solution_order_station = {}

    # For Product-Station Initialization
    # Group by 'PRODUCT' and 'STATION', then take the first occurrence
    product_station_group = data_df.groupby(['PRODUCT'])['MAJ_STAT'].first().reset_index()
    initial_solution_product_station = {(row['PRODUCT'], row['MAJ_STAT']): 1 for index, row in product_station_group.iterrows()}

    # For Order-Station Initialization
    # Group by 'ORDER' and 'STATION', then take the first occurrence
    order_station_group = data_df.groupby(['ORDER'])['MAJ_STAT'].first().reset_index()
    initial_solution_order_station = {(row['ORDER'], row['MAJ_STAT']): 1 for index, row in order_station_group.iterrows()}

    
    return initial_solution_product_station, initial_solution_order_station

In [40]:
def generate_mip_start(data_df):
    # Approach is to initialize initial_solution_product_station and initial_solution_order_station
    initial_solution_product_station = {}
    initial_solution_order_station = {}

    # For Product-Station Initialization
    # Group by 'PRODUCT' and 'STATION', then take the first occurrence
    product_station_group = data_df.groupby(['PRODUCT'])['STATION'].first().reset_index()
    initial_solution_product_station = {(row['PRODUCT'], row['STATION']): 1 for index, row in product_station_group.iterrows()}

    # For Order-Station Initialization
    # Group by 'ORDER' and 'STATION', then take the first occurrence
    order_station_group = data_df.groupby(['ORDER'])['STATION'].first().reset_index()
    initial_solution_order_station = {(row['ORDER'], row['STATION']): 1 for index, row in order_station_group.iterrows()}

    
    return initial_solution_product_station, initial_solution_order_station

# Check the feasibility of the smooth start solution

In [41]:
def check_constraints(initial_solution, products, stations, total_number_of_location_per_station, scaling, speed, df_product):
    violations = []

    # Constraint 1: Each product assigned exactly once
    for p in products:
        assigned_stations = [s for s in stations if initial_solution.get((p, s), 0) == 1]
        if len(assigned_stations) != 1:
            violations.append(f"Product {p} assigned to {len(assigned_stations)} stations, expected exactly 1.")

    # Constraint 2: Total products per station do not exceed allowed locations
    for s in stations:
        total_assigned = sum(initial_solution.get((p, s), 0) for p in products)
        limit = total_number_of_location_per_station.get(s, 0)
        if total_assigned > limit:
            violations.append(f"Station {s} has {total_assigned} products assigned, exceeding the limit {limit}.")

    # Constraint 3: Workload scaling constraint per station
    for s in stations:
        workload = sum(
            initial_solution.get((p, s), 0) * df_product.loc[p]['Product_Frequency'] / speed[s]
            for p in products
        ) * scaling[s]
        if workload > 1:
            violations.append(f"Station {s} has scaled workload {workload:.3f}, exceeding the maximum allowed (1).")

    return violations


## Column Generation solution check

In [42]:
#CG solution to check feasibility
df_cg_solution = pd.read_csv(r'C:\ermal\notebooks\Correlated_Storage_Assignment_Problem\Notebooks\CSLAP_CG_vs_MILP\Sampled_data_CG_vs_MILP\product_assignment_group_sample_46_CG_CPLEX.csv', usecols=['PRODUCT', 'StationID_P'], dtype={'PRODCUT':object, 'StationID_P':object} )# nrows=700000)

In [43]:
df_cg_solution.rename(columns={'StationID_P':'STATION'},inplace=True)

In [44]:
df_cg_solution_merged = df_cg_solution.merge(data_df_new[['ORDER','PRODUCT']],how='inner',on='PRODUCT')

In [45]:
initial_solution_product_station_cg, initial_solution_order_station_cg = generate_mip_start(df_cg_solution_merged)

In [46]:
product_station_cg_map = df_cg_solution.groupby('STATION')['PRODUCT'].apply(set).to_dict()

In [47]:
violations = check_constraints(
    initial_solution=initial_solution_product_station_cg,
    products=products,
    stations=stations,
    total_number_of_location_per_station=total_number_of_location_per_station_2,
    scaling=scaling,
    speed=speed,
    df_product=df_product
)

if violations:
    print("⚠️ Constraint Violations found in Initial Solution:")
    for v in violations:
        print("-", v)
else:
    print("✅ No violations found. Initial solution feasible according to constraints.")


✅ No violations found. Initial solution feasible according to constraints.


In [48]:
df_order_s2 = df_cg_solution_merged[['ORDER', 'STATION']].drop_duplicates()
df_order_s2['cnt'] = df_order_s2.groupby('ORDER').transform('count')
df_order_s2 = df_order_s2[['ORDER','cnt']].drop_duplicates()
s2 = df_order_s2['cnt'].sum()
m2 = df_order_s2['cnt'].mean()
print(s2, m2)

7410 2.1189591078066914


## Heuristic solution check

In [49]:
#CG solution to check feasibility
df_heuristic_solution = pd.read_csv(r'C:\ermal\notebooks\Correlated_Storage_Assignment_Problem\Notebooks\CSLAP_CG_vs_MILP\r46_heuristic_assignment.csv', dtype={'PRODCUT':object, 'STATION':object} )# nrows=700000)
df_heuristic_solution.rename(columns={"Product":"PRODUCT"},inplace=True)
df_heuristic_solution.drop('ORDER',axis=1,inplace=True)

In [51]:
df_heuristic_solution_merged = df_heuristic_solution.merge(data_df_new[['ORDER','PRODUCT']],how='inner',on='PRODUCT')

In [52]:
initial_solution_product_station_heuristic, initial_solution_order_station_heuristic = generate_mip_start_mapped(df_heuristic_solution_merged)

In [53]:
violations = check_constraints(
    initial_solution=initial_solution_product_station_heuristic,
    products=products,
    stations=stations_mapped,
    total_number_of_location_per_station=total_number_of_location_per_station_mapped_2,
    scaling=scaling_mapped,
    speed=speed_mapped,
    df_product=df_product
)

if violations:
    print("⚠️ Constraint Violations found in Initial Solution:")
    for v in violations:
        print("-", v)
else:
    print("✅ No violations found. Initial solution feasible according to constraints.")


⚠️ Constraint Violations found in Initial Solution:
- Station S_1 has scaled workload 2.143, exceeding the maximum allowed (1).
- Station S_3 has scaled workload 1.123, exceeding the maximum allowed (1).
- Station S_5 has scaled workload 1.570, exceeding the maximum allowed (1).
- Station S_9 has scaled workload 1.100, exceeding the maximum allowed (1).
- Station S_11 has scaled workload 1.483, exceeding the maximum allowed (1).
- Station S_12 has scaled workload 1.184, exceeding the maximum allowed (1).


In [54]:
df_order_s2 = df_heuristic_solution_merged[['ORDER', 'MAJ_STAT']].drop_duplicates()
df_order_s2['cnt'] = df_order_s2.groupby('ORDER').transform('count')
df_order_s2 = df_order_s2[['ORDER','cnt']].drop_duplicates()
s2 = df_order_s2['cnt'].sum()
m2 = df_order_s2['cnt'].mean()
print(s2, m2)

7444 2.1286817271947385


In [63]:
product_station_heuristic_map = df_heuristic_solution.groupby('MAJ_STAT')['PRODUCT'].apply(set).to_dict()

In [67]:
for s in stations_mapped:
    print('Station',s)
    print(len(product_station_heuristic_map[s]))
    print(total_number_of_location_per_station_mapped_2[s])

Station S_1
2
2
Station S_2
6
6
Station S_3
4
4
Station S_4
2
2
Station S_5
9
9
Station S_6
3
3
Station S_7
5
5
Station S_8
6
6
Station S_9
2
2
Station S_10
2
2
Station S_11
3
3
Station S_12
1
1
Station S_13
1
1


# Formulating and solving the model for the main data

In [ ]:
# Initialize Hexaly optimizer
with hexaly.HexalyOptimizer() as optimizer:
    optimizer.param.time_limit = 10000
    model = optimizer.model
    optimizer.param.verbosity = 1  
    optimizer.param.nb_threads = 16
    optimizer.param.set_time_between_displays(60)

    #optimizer.add_callback(hexaly.HxCallbackType.DISPLAY, display_callback)

    # Define decision variables
    x = {(p, s): model.bool() for p in products for s in stations}
    #z = {(o, s): model.float(0,1) for o in orders for s in stations}


    # Set constraints
    for p in tqdm(products, desc='Constraint Product', leave=True):
        model.constraint(model.sum(x[p, s] for s in stations) == 1)

    for s in tqdm(stations, desc='Constraint Stations', leave=True):
        model.constraint(model.sum(x[p, s] for p in products) <= total_number_of_location_per_station_2.get(s, 0))
        # Scaled workload constraint:
        model.constraint(
            scaling[s] * model.sum(x[p, s] * df_product.loc[p]['Product_Frequency'] / speed[s]
                                   for p in products) <= 1
        )

    # Progress bar for inner loop
    pbar = tqdm(total=len(orders) * len(stations), desc='Inner loop', leave=True)

    # Define the objective function
    objective = 0
    z = {}
    for o in orders:
        prods_in_o = df_order.loc[o]['PRODUCT_LIST']
        for s in stations:
            z[o, s] = 0
            for p in prods_in_o:
                #z[o, s] = model.or_(x[p, s], z[o, s])  # Logical OR operation
                z[o, s] = model.max(x[p, s], z[o, s])
            objective += z[o, s]
            pbar.update(1)
    model.minimize(objective)
    model.close()

    # Generate the initial solution
    initial_solution_product_station, initial_solution_order_station = generate_mip_start(df_cg_solution_merged)

    # Assign initial solutions to the decision variables
    for (p, s), value in tqdm(initial_solution_product_station.items()):
        if (p, s) in x:
            x[p, s].value = value  # Use the 'value' attribute to set initial values
    # for (o, s), value in tqdm(initial_solution_order_station.items()):
    #     if (o, s) in z:
    #         z[o, s].value = value  
    # print("Solver ready to start")
    # Solve the model
    optimizer.solve()

    # Retrieve results
    results_x = {(p, s): x[p, s].value for p in products for s in stations}

# Output the total runtime
#print(f"Total runtime: {time() - start_time} seconds")

# Retrive the assignment of products to station that the solver has made

In [ ]:
x_0 = np.empty((0,2))
for (p, s), val in results_x.items():
            if val:
                x_0 = np.vstack((x_0, [[p, s]]))

In [ ]:
print(len(x_0))

# Format the assignment of products in the form of a data frame and save it

In [ ]:
product_station_df = pd.DataFrame(x_0, columns=['PRODUCT', 'StationID_P']).drop_duplicates()
#product_station_cg_df = pd.DataFrame(x_initial_cg_cplex, columns=['PRODUCT', 'StationID_P']).drop_duplicates()

#order_station_df = pd.DataFrame(z_0, columns=['ORDER', 'StationID_O']).drop_duplicates()
product_station_df.to_csv(r'C:\ermal\notebooks\Correlated_Storage_Assignment_Problem\Notebooks\CSLAP_CG_vs_MILP\Sampled_data_CG_vs_MILP\product_assignment_group_sample_46_MILP_Hexaly.csv')

#product_station_df.to_csv(r'C:\ermal\notebooks\Correlated_Storage_Assignment_Problem\Notebooks\LocalSolver_version\Group_Results_1\product_assignment_group_localSolver_7_1.csv')
#order_station_df.to_csv(r'C:\ermal\notebooks\Correlated_Storage_Assignment_Problem\Notebooks\LocalSolver_version\Group_Results_1\order_assignment_group_localSolver_1.csv')

In [ ]:
end_time = time() - start_time
print(end_time)

# Check if the workload constraint is respected on the final assignment

In [ ]:
# for s in stations:
#     # Calculate and print the value of the length constraint
#     length_constraint_value = sum(
#         results_x[p, s] * df_product[df_product['PRODUCT'] == p]['Product_Frequency'].values[0] / speed[s]
#         for p in products 
#     )
#     print(f"Length constraint value = {length_constraint_value}, Limit = {total_length[s]}, on station {s}")

# Analyse the assignment and comparisons with initial assignment

In [ ]:
# stations_not_included = ['01.Z8']
# #Filter and count occurrences at the same time
# data_df_initial = data_df_initial[~(data_df_initial['STATION'].isin(stations_not_included))]
# data_df_initial.loc[data_df_initial['STATION'] == '01.GE4', 'STATION'] = '01.E4'

# data_df_initial['Product_Frequency'] = data_df_initial.groupby('PRODUCT')['ORDER'].transform('count')

In [ ]:
df_final = pd.merge(data_df_new,product_station_df, on='PRODUCT',how='left')[['PRODUCT','STATION','StationID_P','ORDER','Product_Frequency']]
#df_final_cg = pd.merge(data_df_new,product_station_df, on='PRODUCT',how='left')[['PRODUCT','STATION','StationID_P','ORDER','Product_Frequency']]
df_final['StationID_P'].fillna(df_final['STATION'], inplace=True)
#df_final_cg['StationID_P'].fillna(df_final_cg['STATION'], inplace=True)
print(df_final)

# Total and Mean number of order station passe

## Initial assignment

In [ ]:
df_order_s1 = df_final[['STATION','ORDER']].drop_duplicates()
df_order_s1['cnt'] = df_order_s1.groupby('ORDER')['STATION'].transform('nunique')
df_order_s1_cnt = df_order_s1[['cnt','ORDER']].drop_duplicates()
s1 = df_order_s1_cnt['cnt'].sum()
m1 = df_order_s1_cnt['cnt'].mean()
print(s1,m1)

## Solver assignment

In [ ]:
df_order_s2 = df_final[['StationID_P','ORDER']].drop_duplicates()
df_order_s2['cnt'] = df_order_s2.groupby('ORDER')['StationID_P'].transform('nunique')
df_order_s2_cnt = df_order_s2[['cnt','ORDER']].drop_duplicates()
s2 = df_order_s2_cnt['cnt'].sum()
m2 = df_order_s2_cnt['cnt'].mean()
print(s2,m2)

In [ ]:
df_order_s1_cnt = df_order_s1_cnt['cnt'].value_counts(dropna=False)
df_order_s2_cnt = df_order_s2_cnt['cnt'].value_counts(dropna=False)

# Function to visualise the number of orders that pass through a respective total number of stations

In [ ]:
# Align the indices
def graph_1(df_order_s1_cnt,df_order_s2_cnt):
    all_indices = sorted(set(df_order_s1_cnt.index) | set(df_order_s2_cnt.index))  # Union of indices from both datasets
    df_order_s1_cnt = df_order_s1_cnt.reindex(all_indices).fillna(0)  # Reindex and fill missing values with 0
    df_order_s2_cnt = df_order_s2_cnt.reindex(all_indices).fillna(0)  # Reindex and fill missing values with 0
    
    print(all_indices)
    # Set up the figure and axis
    plt.figure(figsize=(10, 5))
    ax = plt.gca()
    
    
    # Width of the bars
    width = 0.35
    
    
    # Positions for the bars
    indices = np.arange(len(all_indices))
    
    
    # Plotting
    rects1 = ax.bar(indices - width/2, df_order_s1_cnt, width, label='OLD', alpha=0.6)
    rects2 = ax.bar(indices + width/2, df_order_s2_cnt, width, label='NEW', alpha=0.6)
    
    
    # Add labels and title
    plt.xlabel('Number of Stations')
    plt.ylabel('Number of Orders')
    plt.title('Number of Orders per Stations Numbers')
    
    
    # Add x-ticks and labels
    plt.xticks(indices, all_indices)
    # Add a legend
    plt.legend()
    #plt.savefig(NAME_OF_RUN+'_'+'number_of_orders_per_station_numbers.png')
    
    
    plt.show()

In [ ]:
graph_1(df_order_s1_cnt,df_order_s2_cnt)

# Function to visualise the workload on each station

In [ ]:
def plot_final(tmp_final,xlabel,ylabel,title, sort_by='STATION', ascending=True):


    # Set up the figure and axis
    plt.figure(figsize=(10, 5))
    ax = plt.gca()


    # Width of the bars
    width = 0.35
    
    # Sorting tmp_final based on the specified sort_by column
    tmp_final_sorted = tmp_final.sort_values(by=sort_by, ascending=ascending)


    # Positions for the bars
    stations = tmp_final['STATION'].astype(str).values
    indices = np.arange(len(stations))


    # Plotting
    rects1 = ax.bar(indices - width/2, tmp_final['cnt_lines'], width, label='OLD', alpha=0.6)
    rects2 = ax.bar(indices + width/2, tmp_final['cnt_lines_New'], width, label='NEW', alpha=0.6)


    # Add labels and title
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)


    # Add x-ticks and labels
    plt.xticks(indices, stations, rotation=45)


    # Add a legend
    plt.legend()


    #plt.savefig(NAME_OF_RUN+'_'+'number_of_lines_per_station.png')
    plt.show()

In [ ]:
df_order_lines_plot = df_final.copy()
df_order_lines_plot['cnt_lines_New'] = df_order_lines_plot[['PRODUCT','StationID_P']].groupby('StationID_P').transform('count')
df_order_lines_plot['cnt_lines'] = df_order_lines_plot[['PRODUCT','STATION']].groupby('STATION').transform('count')
tmp_lines = df_order_lines_plot[['StationID_P','cnt_lines_New']].drop_duplicates()
tmp_lines.rename(columns={'StationID_P':'STATION'}, inplace=True)
tmp_lines = tmp_lines.merge(df_order_lines_plot[['STATION','cnt_lines']].drop_duplicates(), on='STATION', how='outer').fillna(0)
tmp_lines.sort_values(by='cnt_lines', ascending=True, inplace=True)
xlabel = 'Station ID'
ylabel = 'Number of lines'
title = 'Number of Lines per Station'
plot_final(tmp_lines, xlabel, ylabel, title, sort_by='cnt_lines', ascending=False)

# Visualising the number of products assigned on each station

In [ ]:
df_order_ref_plot = df_final.copy()
df_order_ref_plot['cnt_lines_New'] = df_order_ref_plot[['PRODUCT','StationID_P']].groupby('StationID_P').transform('nunique')
df_order_ref_plot['cnt_lines'] = df_order_ref_plot[['PRODUCT','STATION']].groupby('STATION').transform('nunique')
tmp_refs = df_order_ref_plot[['StationID_P','cnt_lines_New']].drop_duplicates()
tmp_refs.rename(columns={'StationID_P':'STATION'}, inplace=True)
tmp_refs = tmp_refs.merge(df_order_ref_plot[['STATION','cnt_lines']].drop_duplicates(), on='STATION', how='outer').fillna(0)
tmp_refs.sort_values(by='cnt_lines', ascending=True, inplace=True)
xlabel = 'Station ID'
ylabel = 'Number of lines'
title = 'Number of Lines per Station'
plot_final(tmp_refs, xlabel, ylabel, title, sort_by='cnt_lines', ascending=False)

In [ ]:
import pickle

# Save to a pickle file
with open('x_Hexaly_1.pkl', 'wb') as f:
    pickle.dump(x_0, f)

In [ ]:
print(tmp_refs)

In [ ]:
product_frequency = df_final[['PRODUCT','Product_Frequency']].drop_duplicates().set_index('PRODUCT')['Product_Frequency'].to_dict()
station_product_initial = df_final.groupby('STATION')['PRODUCT'].apply(set).to_dict()
station_product_solution = df_final.groupby('StationID_P')['PRODUCT'].apply(set).to_dict()

In [ ]:
total_length_initial={s:0 for s in stations}
total_length_solution={s:0 for s in stations}

for s in stations:
    for p1 in station_product_initial[s]:
        total_length_initial[s] += product_frequency[p1]/speed[s]
        
    for p2 in station_product_solution[s]:
        total_length_solution[s] += product_frequency[p2]/speed[s]

    print(f"Length constraint value = {total_length_solution[s]}, Limit = {total_length_initial[s]}, on station {s}")

In [ ]:
print(np.sum(total_length_solution[s] for s in stations))

In [ ]:
print(np.sum(total_length_initial[s] for s in stations))